In [10]:
import pandas as pd
import numpy as np
import os
import dabest
from dabest._stats_tools.confint_1group import summary_ci_1group

import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [11]:
laptop = "C:\\Users\\lnico"
workcomp = "C:\\Users\\User"
homecomp = "D:"
titledpath = homecomp

base_path = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Trumelan\\2. Processed\\"
specifiedpath = titledpath + base_path
savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\Appendixstats\\Trumelan\\"

maxtime = 3600

# Responder display name mapping
responder_display_map = {
    "eOPN3": "AsOPN3",
    "eOPN3_ATR": "AsOPN3 [ATR]",
    "eOPN3-TS-ER": "AsOPN3-TS-ER",
    "ACR": "GtACR1",
    "ACR_ATR": "GtACR1 [ATR]",
    "PdCO": "PdCO",
}

def get_responder_display(responder):
    return responder_display_map.get(responder, responder)

In [12]:
def get_phases(light_duration_s=5.0):
    baseline = 60 * 2
    light_dur = light_duration_s * 2
    light_off = baseline + light_dur

    first_phase = slice(0, baseline - 0.5)
    light_on = slice(baseline, baseline + light_dur - 0.5)
    first_min = slice(light_off, light_off + 120 - 0.5)
    third_min = slice(light_off + 240, light_off + 360 - 0.5)
    fifth_min = slice(light_off + 480, light_off + 600 - 0.5)

    return first_phase, light_on, first_min, third_min, fifth_min

In [13]:
def thesis_hedgesg_shared_control(df_dabest, idx_tuple, phase_group_map, metric_name, driver, responder):
    display_responder = get_responder_display(responder)
    
    db = dabest.load(df_dabest, idx=idx_tuple)
    results = db.hedges_g.results
    
    all_cols = list(idx_tuple)
    control_col = all_cols[0]
    test_cols = all_cols[1:]
    
    genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
    genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
    
    rows = []
    for col_name in all_cols:
        phase, group = phase_group_map[col_name]
        col_data = df_dabest[col_name].dropna().values
        
        group_stats = summary_ci_1group(x=col_data, func=np.mean, resamples=5000, alpha=0.05)
        
        mean_val = round(group_stats['summary'], 2)
        mean_ci_low = round(group_stats['bca_ci_low'], 2)
        mean_ci_high = round(group_stats['bca_ci_high'], 2)
        sample_size = len(col_data)
        genotype = genotype_control if group == "Control" else genotype_test
        
        if col_name in test_cols:
            res_idx = test_cols.index(col_name)
            es_val = round(results.difference.iloc[res_idx], 2)
            es_ci_low = round(results.bca_low.iloc[res_idx], 2)
            es_ci_high = round(results.bca_high.iloc[res_idx], 2)
            delta_object = "Hedges' g"
        else:
            es_val = " "
            es_ci_low = " "
            es_ci_high = " "
            delta_object = " "
        
        rows.append({
            "Driver": driver,
            "Responder": display_responder,
            "Group": group,
            "Phase": phase,
            "Genotype": genotype,
            "Sample Size": sample_size,
            "Mean": mean_val,
            "Mean_CI_low": mean_ci_low,
            "Mean_CI_high": mean_ci_high,
            "Effect Size": es_val,
            "Effect Size_CI_low": es_ci_low,
            "Effect Size_CI_high": es_ci_high,
            "Delta Object": delta_object,
            "Metric": metric_name,
        })
    
    return pd.DataFrame(rows)

## Intensity thesis stats

In [14]:
lst = ["elav", "vGAT"]
responder = "eOPN3_ATR"
w1118 = "w1118"

first_phase, light_on, first_min, third_min, fifth_min = get_phases(5.0)

intensity_idx = ("No Light_control", "No Light_expt", "Quarter_expt", "Half_expt", "Full_expt")

intensity_phase_group = {
    'No Light_control': ('No Light', 'Control'),
    'No Light_expt': ('No Light', 'Test'),
    'Quarter_expt': ('Quarter', 'Test'),
    'Half_expt': ('Half', 'Test'),
    'Full_expt': ('Full', 'Test'),
}

for driver in lst:
    print(driver)
    
    intensity_files_line = {
        'No Light_control': specifiedpath + w1118 + " x " + responder + "_0s",
        'Quarter_control': specifiedpath + w1118 + " x " + responder + "_quarter_5s",
        'Half_control': specifiedpath + w1118 + " x " + responder + "_half_5s",
        'Full_control': specifiedpath + w1118 + " x " + responder + "_5s",
        
        'No Light_expt': specifiedpath + driver + " x " + responder + "_0s",
        'Quarter_expt': specifiedpath + driver + " x " + responder + "_quarter_5s",
        'Half_expt': specifiedpath + driver + " x " + responder + "_half_5s",
        'Full_expt': specifiedpath + driver + " x " + responder + "_5s",
    }

    df_dabest_speed = pd.DataFrame()
    df_dabest_activity = pd.DataFrame()

    for label, filepath in intensity_files_line.items():
        df = pd.read_csv(filepath + "_max" + str(maxtime) + "s.csv").set_index('Time (s)')
        df = df.loc[first_min, :]

        df_dabest_speed[label] = df.filter(regex="^Speed").T.reset_index(drop=True).mean(axis=1)
        df_dabest_activity[label] = df.filter(regex="^ActivityLevel").T.reset_index(drop=True).mean(axis=1)

    df_intensity_thesis = pd.concat([
        thesis_hedgesg_shared_control(df_dabest_speed, intensity_idx, intensity_phase_group, "speed", driver, responder),
        thesis_hedgesg_shared_control(df_dabest_activity, intensity_idx, intensity_phase_group, "activitylevel", driver, responder),
    ], ignore_index=True)

    df_intensity_thesis['Assay'] = 'Intensity'
    df_intensity_thesis.to_csv(savedir + driver + " x " + responder + "_intensity_thesis_stats.csv", index=False)

print("Done!")

elav
vGAT
Done!


## Duration thesis stats

In [15]:
lst = ["elav", "vGAT"]
responder = "eOPN3_ATR"
w1118 = "w1118"

duration_idx = ("0s_control", "0s_expt", "5s_expt", "30s_expt", "60s_expt")

duration_phase_group = {
    '0s_control': ('0s', 'Control'),
    '0s_expt': ('0s', 'Test'),
    '5s_expt': ('5s', 'Test'),
    '30s_expt': ('30s', 'Test'),
    '60s_expt': ('60s', 'Test'),
}

for driver in lst:
    print(driver)
    
    duration_files_line = {
        '0s_control': specifiedpath + w1118 + " x " + responder + "_0s",
        '5s_control': specifiedpath + w1118 + " x " + responder + "_5s",
        '30s_control': specifiedpath + w1118 + " x " + responder + "_30s",
        '60s_control': specifiedpath + w1118 + " x " + responder + "_60s",
        
        '0s_expt': specifiedpath + driver + " x " + responder + "_0s",
        '5s_expt': specifiedpath + driver + " x " + responder + "_5s",
        '30s_expt': specifiedpath + driver + " x " + responder + "_30s",
        '60s_expt': specifiedpath + driver + " x " + responder + "_60s",
    }

    df_dabest_speed = pd.DataFrame()
    df_dabest_activity = pd.DataFrame()

    for label, filepath in duration_files_line.items():
        duration = int(label.split('s_')[0])
        if duration == 0:
            duration = 60
        first_phase, light_on, first_min, third_min, fifth_min = get_phases(duration)
        
        df = pd.read_csv(filepath + "_max" + str(maxtime) + "s.csv").set_index('Time (s)')
        df = df.loc[first_min, :]

        df_dabest_speed[label] = df.filter(regex="^Speed").T.reset_index(drop=True).mean(axis=1)
        df_dabest_activity[label] = df.filter(regex="^ActivityLevel").T.reset_index(drop=True).mean(axis=1)

    df_duration_thesis = pd.concat([
        thesis_hedgesg_shared_control(df_dabest_speed, duration_idx, duration_phase_group, "speed", driver, responder),
        thesis_hedgesg_shared_control(df_dabest_activity, duration_idx, duration_phase_group, "activitylevel", driver, responder),
    ], ignore_index=True)

    df_duration_thesis['Assay'] = 'Duration'
    df_duration_thesis.to_csv(savedir + driver + " x " + responder + "_duration_thesis_stats.csv", index=False)

print("Done!")

elav
vGAT
Done!
